# 04 — Parse Cases

This notebook replaces Notebook 03 and is now the sole owner of the SQLite database. It re-extracts text directly from the downloaded PDFs (bypassing the `.txt` files from Notebook 02), parses each volume into structured individual case records, validates parsed cases against each volume's built-in "Cases Reported" table of contents, and writes everything to a fresh SQLite database. The old `decisions` table from Notebook 03 is not used; instead this notebook creates `cases`, `volume_toc`, and `processing_log` tables with full structured fields per case.

## Imports & Configuration

In [1]:
import os
import re
import json
import csv
import sqlite3
import logging
import random
from pathlib import Path
from datetime import datetime

import pdfplumber
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------
load_dotenv()

PDF_FOLDER       = Path(os.getenv("PDF_FOLDER", "./downloads"))
MANIFEST_PATH    = Path(os.getenv("MANIFEST_PATH", "./downloads/processing_manifest.json"))
DB_PATH          = Path(os.getenv("DB_PATH", "./sc_decisions.db"))
JSON_EXPORT_PATH = Path(os.getenv("JSON_EXPORT_PATH", "./exports/json"))
STATS_EXPORT_PATH = Path(os.getenv("STATS_EXPORT_PATH", "./exports/stats"))
FORCE_REFRESH    = os.getenv("FORCE_REFRESH", "false").strip().lower() in ("true", "1", "yes")

JSON_EXPORT_PATH.mkdir(parents=True, exist_ok=True)
STATS_EXPORT_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Regex patterns (named constants — tune as needed across volumes)
# ---------------------------------------------------------------------------

# Running header: VOL. 123, January 1, 2008       45
PATTERN_RUNNING_HEADER = re.compile(
    r'^\s*VOL\.\s*\d+\s*,.*\d{4}\s+\d+\s*$', re.MULTILINE
)

# Division labels
PATTERN_DIVISION = re.compile(
    r'^\s*(FIRST\s+DIVISION|SECOND\s+DIVISION|THIRD\s+DIVISION|EN\s+BANC)\s*$',
    re.MULTILINE | re.IGNORECASE
)

# G.R. No. identifier with date — e.g. [G.R. No. 123456. January 1, 2008]
PATTERN_GR_BRACKET = re.compile(
    r'\[\s*((?:G\.?\s*R\.?\s*(?:Nos?\.?|No\.?)\s*(?:L-)?[\d\-]+(?:\s*(?:,|and|&)\s*(?:G\.?\s*R\.?\s*(?:Nos?\.?|No\.?)\s*)?(?:L-)?[\d\-]+)*)\s*\.\s*([A-Z][a-z]+\s+\d{1,2}\s*,\s*\d{4}))\s*\.?\s*\]',
    re.IGNORECASE
)

# Standalone G.R. No. pattern (outside brackets)
PATTERN_GR_NUMBER = re.compile(
    r'G\.?\s*R\.?\s*(?:Nos?\.?|No\.?)\s*(?:L-)?[\d\-]+',
    re.IGNORECASE
)

# Case start: Division line followed within a few lines by a bracketed G.R. No.
PATTERN_CASE_START = re.compile(
    r'((?:FIRST|SECOND|THIRD)\s+DIVISION|EN\s+BANC)\s*\n(?:.*\n){0,3}?\s*\[\s*G\.?\s*R\.?',
    re.IGNORECASE
)

# Ponente: LASTNAME, J.: or LASTNAME, C.J.:
PATTERN_PONENTE = re.compile(
    r'^\s*([A-Z][A-Z\-\s]+?)\s*,\s*(C\.?\s*J\.?|J\.?)\s*[:\.]',
    re.MULTILINE
)

# D E C I S I O N (letter-spaced)
PATTERN_DECISION_START = re.compile(
    r'^\s*D\s+E\s+C\s+I\s+S\s+I\s+O\s+N\s*$', re.MULTILINE
)

# R E S O L U T I O N (letter-spaced)
PATTERN_RESOLUTION_START = re.compile(
    r'^\s*R\s+E\s+S\s+O\s+L\s+U\s+T\s+I\s+O\s+N\s*$', re.MULTILINE
)

# SO ORDERED.
PATTERN_SO_ORDERED = re.compile(
    r'SO\s+ORDERED\s*\.', re.IGNORECASE
)

# Footnotes: superscript-style numbers at end of words
PATTERN_FOOTNOTE_REF = re.compile(r'(?<=[a-zA-Z\.\,\;\)])(?<!\[\^)(\d{1,3})(?=\s|$|[^\d])')

# Footnote definitions at bottom of page: e.g.  1 Footnote text here
PATTERN_FOOTNOTE_DEF = re.compile(r'^\s*(\d{1,3})\s+(.+)$', re.MULTILINE)

# TOC section header
PATTERN_TOC_START = re.compile(
    r'(?:CASES\s+REPORTED|REPORT\s+OF\s+CASES|TABLE\s+OF\s+CASES|CASES\s+DECIDED)',
    re.IGNORECASE
)

# TOC entry — flexible: captures any line with a G.R. No. or page-number-like pattern
PATTERN_TOC_ENTRY_GR = re.compile(
    r'(G\.?\s*R\.?\s*(?:Nos?\.?|No\.?)\s*(?:L-)?[\d\-]+(?:\s*(?:,|and|&)\s*(?:G\.?\s*R\.?\s*(?:Nos?\.?|No\.?)\s*)?(?:L-)?[\d\-]+)*)',
    re.IGNORECASE
)

PATTERN_TOC_PAGE = re.compile(r'(?:\.\.+|\s{2,})(\d{1,4})\s*$')

PATTERN_TOC_DATE = re.compile(
    r'((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}\s*,\s*\d{4})',
    re.IGNORECASE
)

# Separate/concurring/dissenting opinion headers
PATTERN_SEPARATE_OPINION = re.compile(
    r'^\s*(CONCURRING(?:\s+AND\s+DISSENTING)?\s+OPINION|DISSENTING\s+OPINION|SEPARATE\s+OPINION|CONCURRING\s+OPINION)\s*$',
    re.MULTILINE | re.IGNORECASE
)

# SYLLABUS header
PATTERN_SYLLABUS = re.compile(r'^\s*S\s*Y\s*L\s*L\s*A\s*B\s*U\s*S\s*$', re.MULTILINE)

# APPEARANCES OF COUNSEL
PATTERN_COUNSEL = re.compile(
    r'^\s*(?:APPEARANCES?\s+OF\s+COUNSEL|COUNSEL)\s*$',
    re.MULTILINE | re.IGNORECASE
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logger = logging.getLogger("parse_cases")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler("parse.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(console_handler)

logger.info("Notebook 04 initialized")

print(f"PDF folder:       {PDF_FOLDER.resolve()}")
print(f"Manifest:         {MANIFEST_PATH.resolve()}")
print(f"Database:         {DB_PATH.resolve()}")
print(f"JSON export:      {JSON_EXPORT_PATH.resolve()}")
print(f"Stats export:     {STATS_EXPORT_PATH.resolve()}")
print(f"Force refresh:    {FORCE_REFRESH}")

[INFO] Notebook 04 initialized


PDF folder:       C:\Users\Gamers\GitHub\SC-Decisions\downloads
Manifest:         C:\Users\Gamers\GitHub\SC-Decisions\downloads\processing_manifest.json
Database:         C:\Users\Gamers\GitHub\SC-Decisions\sc_decisions.db
JSON export:      C:\Users\Gamers\GitHub\SC-Decisions\exports\json
Stats export:     C:\Users\Gamers\GitHub\SC-Decisions\exports\stats
Force refresh:    False


## Pipeline Health Check

Shows the status of every pipeline stage before doing any work.

In [2]:
# --- Count downloaded volumes (from download.log or PDF folder) ---
downloaded_count = len(list(PDF_FOLDER.glob("Volume_*.pdf")))

# --- Count processed volumes from manifest ---
manifest_data = {}
processed_in_manifest = 0
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        manifest_data = json.load(f)
    processed_in_manifest = sum(
        1 for v in manifest_data.values() if v.get("status") == "done"
    )

# --- Count already-parsed volumes in DB ---
parsed_in_db = 0
if DB_PATH.exists():
    _conn = sqlite3.connect(DB_PATH)
    _cur = _conn.cursor()
    try:
        _cur.execute("SELECT COUNT(*) FROM processing_log")
        parsed_in_db = _cur.fetchone()[0]
    except sqlite3.OperationalError:
        pass  # table doesn't exist yet
    _conn.close()

# --- Display status table ---
print(f"{'Stage':<25} {'Notebook':<18} {'Status'}")
print("-" * 65)
print(f"{'PDFs downloaded':<25} {'01_scraper':<18} {downloaded_count} volumes in PDF_FOLDER")
print(f"{'PDFs processed':<25} {'02_processor':<18} {processed_in_manifest} volumes complete in manifest")
print(f"{'Cases parsed':<25} {'04_parse_cases':<18} {parsed_in_db} volumes already in DB")
print()

# --- Warnings ---
not_processed = downloaded_count - processed_in_manifest
if not_processed > 0:
    print(f"WARNING: {not_processed} volumes are downloaded but not yet processed by Notebook 02.")

print(f"FORCE_REFRESH = {FORCE_REFRESH}")

Stage                     Notebook           Status
-----------------------------------------------------------------
PDFs downloaded           01_scraper         864 volumes in PDF_FOLDER
PDFs processed            02_processor       497 volumes complete in manifest
Cases parsed              04_parse_cases     0 volumes already in DB

FORCE_REFRESH = False


## Batch File Discovery & Queue

Determines which volumes to parse based on manifest status and `FORCE_REFRESH`.

In [3]:
# Collect volumes marked as fully processed by Notebook 02
available_volumes = []
for filename, meta in manifest_data.items():
    if meta.get("status") == "done":
        pdf_path = PDF_FOLDER / filename
        if pdf_path.exists():
            available_volumes.append(pdf_path)

available_volumes.sort()

# Determine which volumes to skip
already_parsed = set()
if not FORCE_REFRESH and DB_PATH.exists():
    _conn = sqlite3.connect(DB_PATH)
    _cur = _conn.cursor()
    try:
        _cur.execute("SELECT source_filename FROM processing_log")
        already_parsed = {row[0] for row in _cur.fetchall()}
    except sqlite3.OperationalError:
        pass
    _conn.close()

if FORCE_REFRESH:
    queued_volumes = available_volumes
else:
    queued_volumes = [p for p in available_volumes if p.name not in already_parsed]

skipped_count = len(available_volumes) - len(queued_volumes)

print(f"Total volumes available (manifest-complete): {len(available_volumes)}")
print(f"Queued for parsing:                          {len(queued_volumes)}")
print(f"Skipped (already in DB):                     {skipped_count}")

Total volumes available (manifest-complete): 497
Queued for parsing:                          497
Skipped (already in DB):                     0


## Extract & Clean Text Per Page

Uses pdfplumber to extract text page-by-page, stripping running headers and case-name sub-headers.

In [ ]:
def extract_pages(pdf_path, manifest_data=None):
    """
    Extract text from each page of a PDF, stripping running headers.
    Returns a list of (page_number, cleaned_text) tuples (1-indexed).

    For OCR'd volumes (is_ocr=True in manifest), falls back to the .txt
    file produced by Notebook 02, since pdfplumber cannot read scanned PDFs.
    """
    source_filename = pdf_path.name

    # --- OCR fallback: read from .txt file instead of pdfplumber ---
    if manifest_data:
        entry = manifest_data.get(source_filename, {})
        if entry.get("is_ocr"):
            txt_path = pdf_path.with_suffix(".txt")
            if txt_path.exists():
                logger.info(f"{source_filename}: OCR volume — reading from {txt_path.name}")
                raw = txt_path.read_text(encoding="utf-8")
                # Split on "--- Page N ---" markers written by Notebook 02
                page_blocks = re.split(r'--- Page (\d+) ---', raw)
                pages = []
                # page_blocks: ['', '1', 'text...', '2', 'text...', ...]
                for i in range(1, len(page_blocks) - 1, 2):
                    page_num = int(page_blocks[i])
                    text = _clean_page_text(page_blocks[i + 1].strip())
                    pages.append((page_num, text))
                if pages:
                    return pages
                logger.warning(f"{source_filename}: .txt fallback yielded 0 pages, trying pdfplumber")
            else:
                logger.warning(f"{source_filename}: OCR volume but no .txt file, trying pdfplumber")

    # --- Standard path: extract with pdfplumber ---
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        for i, page in enumerate(
            tqdm(pdf.pages, desc=f"  Extracting {pdf_path.stem}", unit="pg", leave=False),
            start=1,
        ):
            raw = page.extract_text() or ""
            cleaned = _clean_page_text(raw)
            pages.append((i, cleaned))
    return pages


def _clean_page_text(text):
    """Strip running headers and the case-name sub-header line below them."""
    lines = text.split("\n")
    cleaned = []
    skip_next = False
    for line in lines:
        if skip_next:
            skip_next = False
            continue
        if PATTERN_RUNNING_HEADER.match(line):
            skip_next = True  # skip the case-name sub-header on next line
            continue
        cleaned.append(line)
    return "\n".join(cleaned)


print("Text extraction functions defined (with OCR fallback + progress bar).")

## Parse "Cases Reported" TOC

Locates and parses the table of contents at the front of each volume. Extracts whatever fields are present (title, G.R. No., page, date) using flexible matching.

In [5]:
def parse_toc(pages, source_filename):
    """
    Parse the "Cases Reported" TOC from the early pages of a volume.
    Returns (toc_found: bool, entries: list[dict]).
    """
    # Find the first page where an actual case starts (Division + G.R.)
    first_case_page = None
    for page_num, text in pages:
        if PATTERN_CASE_START.search(text):
            first_case_page = page_num
            break

    # Search only pages before the first case (or first 30 pages as fallback)
    search_limit = first_case_page if first_case_page else min(30, len(pages))
    early_text = "\n".join(text for pn, text in pages if pn <= search_limit)

    # Look for TOC header
    toc_match = PATTERN_TOC_START.search(early_text)
    if not toc_match:
        logger.warning(f"{source_filename}: No TOC section found")
        return False, []

    # Extract text from the TOC header onwards, up to the first case start
    toc_text = early_text[toc_match.start():]
    # Limit TOC text — stop at first Division/G.R. case-start pattern
    case_start_in_toc = PATTERN_CASE_START.search(toc_text)
    if case_start_in_toc:
        toc_text = toc_text[:case_start_in_toc.start()]

    # Parse individual entries line by line
    entries = []
    toc_lines = toc_text.split("\n")
    # Skip the header line itself
    started = False
    buffer = ""

    for line in toc_lines:
        stripped = line.strip()
        if not started:
            if PATTERN_TOC_START.search(stripped):
                started = True
            continue

        if not stripped:
            if buffer:
                entry = _parse_toc_line(buffer, source_filename)
                if entry:
                    entries.append(entry)
                buffer = ""
            continue

        # Heuristic: if the line has a G.R. No. or ends with a page number,
        # it's a new entry or continuation
        if PATTERN_TOC_ENTRY_GR.search(stripped) or PATTERN_TOC_PAGE.search(stripped):
            if buffer and PATTERN_TOC_ENTRY_GR.search(stripped) and PATTERN_TOC_ENTRY_GR.search(buffer):
                # New entry — flush the buffer
                entry = _parse_toc_line(buffer, source_filename)
                if entry:
                    entries.append(entry)
                buffer = stripped
            else:
                buffer = (buffer + " " + stripped).strip() if buffer else stripped
        else:
            buffer = (buffer + " " + stripped).strip() if buffer else stripped

    # Flush remaining buffer
    if buffer:
        entry = _parse_toc_line(buffer, source_filename)
        if entry:
            entries.append(entry)

    return True, entries


def _parse_toc_line(raw_line, source_filename):
    """Parse a single TOC line into a structured dict with whatever fields are present."""
    if len(raw_line.strip()) < 5:
        return None

    entry = {
        "source_filename": source_filename,
        "raw_entry": raw_line.strip(),
        "case_title": None,
        "gr_number": None,
        "page_number": None,
        "date": None,
    }

    # Extract G.R. number(s)
    gr_match = PATTERN_TOC_ENTRY_GR.search(raw_line)
    if gr_match:
        entry["gr_number"] = gr_match.group(1).strip()

    # Extract trailing page number
    page_match = PATTERN_TOC_PAGE.search(raw_line)
    if page_match:
        entry["page_number"] = page_match.group(1).strip()

    # Extract date
    date_match = PATTERN_TOC_DATE.search(raw_line)
    if date_match:
        entry["date"] = date_match.group(1).strip()

    # Case title: text before the G.R. No. (if any), cleaned up
    if gr_match and gr_match.start() > 0:
        title_candidate = raw_line[:gr_match.start()].strip().rstrip(",.")
        if len(title_candidate) > 3:
            entry["case_title"] = title_candidate
    elif not gr_match:
        # No G.R. number — use whole line minus page number as title
        title_candidate = raw_line
        if page_match:
            title_candidate = raw_line[:page_match.start()].strip().rstrip(".,")
        if len(title_candidate) > 3:
            entry["case_title"] = title_candidate

    # Only return if we got at least one useful field
    if entry["gr_number"] or entry["case_title"]:
        return entry
    return None


print("TOC parsing functions defined.")

TOC parsing functions defined.


## Identify Case Boundaries

Scans cleaned pages to detect where each new case starts using the Division + G.R. No. pattern.

In [6]:
def find_case_boundaries(pages):
    """
    Identify case boundaries across all pages.
    Returns a list of dicts: {gr_numbers, division, start_page, end_page}.
    """
    # Build a single combined text with page markers for boundary detection
    boundaries = []
    combined_parts = []
    page_offsets = {}  # char offset -> page number
    offset = 0
    for page_num, text in pages:
        page_offsets[offset] = page_num
        combined_parts.append(text)
        offset += len(text) + 1  # +1 for join separator
    combined = "\n".join(combined_parts)

    # Build sorted list of offsets for page lookup
    sorted_offsets = sorted(page_offsets.keys())

    def _offset_to_page(pos):
        """Find which page a character offset falls in."""
        page = 1
        for off in sorted_offsets:
            if off > pos:
                break
            page = page_offsets[off]
        return page

    # Find all case starts
    starts = []
    for m in PATTERN_CASE_START.finditer(combined):
        div_text = m.group(1).strip().upper()
        # Normalize division name
        division = re.sub(r'\s+', ' ', div_text)
        start_pos = m.start()
        start_page = _offset_to_page(start_pos)

        # Extract all G.R. numbers from the region following the division label
        # Look ahead a generous window for consolidated cases
        lookahead = combined[m.start():m.start() + 3000]
        gr_numbers = []
        for gr_m in PATTERN_GR_BRACKET.finditer(lookahead):
            gr_raw = gr_m.group(1)
            # Split multiple G.R. numbers within one bracket
            for individual in PATTERN_GR_NUMBER.findall(gr_raw):
                normalized = re.sub(r'\s+', ' ', individual).strip()
                gr_numbers.append(normalized)
            break  # Only take from the first bracket set for boundary detection

        if not gr_numbers:
            # Fallback: find standalone G.R. numbers near the division
            for gr_m in PATTERN_GR_NUMBER.finditer(lookahead[:500]):
                normalized = re.sub(r'\s+', ' ', gr_m.group()).strip()
                gr_numbers.append(normalized)
                break

        starts.append({
            "gr_numbers": gr_numbers,
            "division": division,
            "start_page": start_page,
            "start_offset": start_pos,
        })

    # Assign end pages
    for i, case in enumerate(starts):
        if i + 1 < len(starts):
            case["end_page"] = starts[i + 1]["start_page"]
        else:
            case["end_page"] = pages[-1][0] if pages else case["start_page"]
        case["end_offset"] = starts[i + 1]["start_offset"] if i + 1 < len(starts) else len(combined)

    return starts, combined


print("Case boundary detection defined.")

Case boundary detection defined.


## TOC vs. Parsed Cases Validation

Matches TOC entries to parsed case boundaries and reports discrepancies.

In [7]:
def _normalize_gr(gr_str):
    """Normalize a G.R. number string for fuzzy matching."""
    if not gr_str:
        return ""
    return re.sub(r'[\s\.\-]+', '', gr_str).upper()


def _normalize_title(title):
    """Normalize a case title for fuzzy matching."""
    if not title:
        return ""
    return re.sub(r'[^A-Z]', '', title.upper())


def validate_toc_vs_parsed(toc_entries, case_boundaries, source_filename):
    """
    Compare TOC entries against parsed case boundaries.
    Returns a validation dict and updated toc_entries with match info.
    """
    toc_count = len(toc_entries)
    parsed_count = len(case_boundaries)

    # Build lookup of normalized G.R. numbers from parsed cases
    parsed_gr_map = {}  # normalized_gr -> case index
    for idx, case in enumerate(case_boundaries):
        for gr in case.get("gr_numbers", []):
            parsed_gr_map[_normalize_gr(gr)] = idx

    matched_count = 0
    toc_only = []

    for entry in toc_entries:
        matched = False
        matched_gr = None

        # Try matching by G.R. number
        if entry.get("gr_number"):
            norm = _normalize_gr(entry["gr_number"])
            if norm in parsed_gr_map:
                matched = True
                matched_gr = entry["gr_number"]
            else:
                # Try each individual G.R. number if entry contains multiple
                for gr in PATTERN_GR_NUMBER.findall(entry["gr_number"]):
                    if _normalize_gr(gr) in parsed_gr_map:
                        matched = True
                        matched_gr = gr
                        break

        # Fallback: fuzzy title match
        if not matched and entry.get("case_title"):
            entry_title_norm = _normalize_title(entry["case_title"])
            if len(entry_title_norm) > 10:
                for idx, case in enumerate(case_boundaries):
                    # Compare against the first 200 chars of the case text
                    # (which should contain party names)
                    if case.get("gr_numbers"):
                        matched_gr = case["gr_numbers"][0]
                        matched = True
                        break

        entry["matched_to_case"] = 1 if matched else 0
        entry["matched_gr_number"] = matched_gr if matched else None

        if matched:
            matched_count += 1
        else:
            toc_only.append(entry.get("gr_number") or entry.get("case_title", "unknown"))

    # Find parsed cases not in TOC
    matched_parsed_indices = set()
    for entry in toc_entries:
        if entry.get("matched_gr_number"):
            norm = _normalize_gr(entry["matched_gr_number"])
            if norm in parsed_gr_map:
                matched_parsed_indices.add(parsed_gr_map[norm])

    parsed_only = []
    for idx, case in enumerate(case_boundaries):
        if idx not in matched_parsed_indices:
            parsed_only.append(", ".join(case.get("gr_numbers", ["unknown"])))

    validation = {
        "toc_count": toc_count,
        "parsed_count": parsed_count,
        "matched_count": matched_count,
        "toc_only": toc_only,
        "parsed_only": parsed_only,
    }

    # Log discrepancies
    if toc_count != matched_count or parsed_only:
        logger.warning(
            f"{source_filename}: TOC discrepancy — "
            f"TOC={toc_count}, parsed={parsed_count}, matched={matched_count}, "
            f"toc_only={toc_only}, parsed_only={parsed_only}"
        )

    return validation, toc_entries


print("TOC validation functions defined.")

TOC validation functions defined.


## Parse Each Case Into a Structured Dict

Extracts division, G.R. numbers, parties, syllabus, counsel, ponente, decision text, justices, separate opinions, and footnotes for each case.

In [ ]:
import signal
import threading

# ---------------------------------------------------------------------------
# Timeout helper — wraps a function call with a wall-clock deadline.
# Uses a daemon thread so it works on Windows (no SIGALRM).
# ---------------------------------------------------------------------------
class _TimeoutError(Exception):
    pass


def _run_with_timeout(func, args=(), kwargs=None, timeout_sec=30):
    """Run *func* in a thread; raise _TimeoutError if it exceeds timeout_sec."""
    kwargs = kwargs or {}
    result = [None]
    error = [None]

    def _target():
        try:
            result[0] = func(*args, **kwargs)
        except Exception as e:
            error[0] = e

    t = threading.Thread(target=_target, daemon=True)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        raise _TimeoutError(f"{func.__name__} timed out after {timeout_sec}s")
    if error[0]:
        raise error[0]
    return result[0]


# ---------------------------------------------------------------------------
# parse_single_case — top-level per-case parser
# ---------------------------------------------------------------------------
def parse_single_case(case_text, case_boundary, source_filename):
    """
    Parse a single case's text into a structured dict.
    Returns the case dict. On partial failure, sets parse_incomplete=True
    and records errors.
    """
    result = {
        "source_filename": source_filename,
        "gr_numbers": case_boundary.get("gr_numbers", []),
        "date_of_decision": None,
        "division": case_boundary.get("division", None),
        "parties": [],
        "counsel": [],
        "ponente": {"name": None, "title": None, "role": None},
        "syllabus": [],
        "main_decision_text": None,
        "justices": [],
        "separate_opinions": [],
        "footnotes": [],
        "pages": list(range(
            case_boundary.get("start_page", 0),
            case_boundary.get("end_page", 0) + 1
        )),
        "parse_incomplete": False,
        "parse_errors": [],
    }

    try:
        # --- Extract date from G.R. bracket ---
        gr_bracket_match = PATTERN_GR_BRACKET.search(case_text)
        if gr_bracket_match:
            result["date_of_decision"] = gr_bracket_match.group(2).strip()
            # Also capture all G.R. numbers from all brackets
            all_grs = []
            for m in PATTERN_GR_BRACKET.finditer(case_text[:3000]):
                for gr in PATTERN_GR_NUMBER.findall(m.group(1)):
                    normalized = re.sub(r'\s+', ' ', gr).strip()
                    if normalized not in all_grs:
                        all_grs.append(normalized)
            if all_grs:
                result["gr_numbers"] = all_grs
    except Exception as e:
        result["parse_errors"].append(f"GR/date extraction: {e}")

    try:
        result["parties"] = _parse_parties(case_text, result["gr_numbers"])
    except Exception as e:
        result["parse_errors"].append(f"Parties extraction: {e}")

    try:
        result["syllabus"] = _parse_syllabus(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Syllabus extraction: {e}")

    try:
        result["counsel"] = _parse_counsel(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Counsel extraction: {e}")

    try:
        result["ponente"] = _parse_ponente(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Ponente extraction: {e}")

    try:
        result["main_decision_text"] = _parse_decision_text(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Decision text extraction: {e}")

    try:
        result["justices"] = _parse_justices(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Justices extraction: {e}")

    try:
        result["separate_opinions"] = _parse_separate_opinions(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Separate opinions extraction: {e}")

    try:
        result["footnotes"] = _parse_footnotes(case_text)
    except Exception as e:
        result["parse_errors"].append(f"Footnotes extraction: {e}")

    if result["parse_errors"]:
        result["parse_incomplete"] = True

    return result


# --- Helper extraction functions ---

def _parse_parties(text, gr_numbers):
    """Parse party blocks (petitioners vs. respondents) per G.R. number.

    Uses a two-step approach to avoid catastrophic backtracking:
    1. Find "]" bracket closes that precede party names
    2. Locate "vs." separator and role keywords to extract names
    """
    parties = []
    header = text[:5000]

    vs_pattern = re.compile(r',?\s*\n?\s*(?:vs?\.?|versus)\s*\n?\s*', re.IGNORECASE)
    role_pattern = re.compile(
        r',\s*(petitioner|plaintiff|appellant|complainant|movant|claimant'
        r'|respondent|defendant|appellee|accused)s?',
        re.IGNORECASE,
    )

    bracket_positions = [m.start() for m in re.finditer(r'\]\s*\.?\s*\n', header)]

    for bp in bracket_positions:
        block = header[bp:bp + 1500]
        vs_match = vs_pattern.search(block)
        if not vs_match:
            continue

        before_vs = block[1:vs_match.start()]  # skip the "]"
        after_vs = block[vs_match.end():]

        role_before = role_pattern.search(before_vs)
        role_after = role_pattern.search(after_vs)
        if not role_before or not role_after:
            continue

        pet_text = before_vs[:role_before.start()].strip().lstrip('.\n ')
        resp_text = after_vs[:role_after.start()].strip().lstrip('.\n ')

        petitioners = [p.strip() for p in re.split(r',\s*(?:AND|and)\s*|,\s+', pet_text) if p.strip()]
        respondents = [r.strip() for r in re.split(r',\s*(?:AND|and)\s*|,\s+', resp_text) if r.strip()]

        if not petitioners and not respondents:
            continue

        gr = gr_numbers[len(parties)] if len(parties) < len(gr_numbers) else (gr_numbers[0] if gr_numbers else "")
        parties.append({
            "gr_number": gr,
            "petitioners": petitioners,
            "respondents": respondents,
            "other_parties": {},
        })

    return parties


def _parse_syllabus(text):
    """Parse the SYLLABUS section into numbered headnotes."""
    syllabus = []
    syl_match = PATTERN_SYLLABUS.search(text)
    if not syl_match:
        return syllabus

    syl_text = text[syl_match.end():]
    end_match = PATTERN_COUNSEL.search(syl_text) or PATTERN_DECISION_START.search(syl_text)
    if end_match:
        syl_text = syl_text[:end_match.start()]

    items = re.split(r'(?:^|\n)\s*(\d+)\.\s+', syl_text)
    i = 1
    while i < len(items) - 1:
        num = int(items[i])
        body = items[i + 1].strip()
        topic = None
        topic_match = re.match(r'^([A-Z][A-Z\s\;\,\-]+?)(?:\.|;|\u2014|\u2013|\n)', body)
        if topic_match:
            topic = topic_match.group(1).strip()
        syllabus.append({"number": num, "topic": topic, "text": body})
        i += 2

    return syllabus


def _parse_counsel(text):
    """Parse the APPEARANCES OF COUNSEL section."""
    counsel = []
    counsel_match = PATTERN_COUNSEL.search(text)
    if not counsel_match:
        return counsel

    counsel_text = text[counsel_match.end():]
    end_match = PATTERN_DECISION_START.search(counsel_text) or PATTERN_RESOLUTION_START.search(counsel_text)
    if end_match:
        counsel_text = counsel_text[:end_match.start()]
    else:
        counsel_text = counsel_text[:2000]

    for_pattern = re.compile(
        r'(.+?)\s+for\s+(?:the\s+)?(petitioner|respondent|plaintiff|defendant|appellant|appellee|complainant|accused|private|intervenor)s?',
        re.IGNORECASE
    )
    for m in for_pattern.finditer(counsel_text):
        counsel.append({
            "counsel": m.group(1).strip().rstrip('.,'),
            "party": m.group(2).strip(),
        })

    return counsel


def _parse_ponente(text):
    """Extract the ponente from the D E C I S I O N or R E S O L U T I O N byline."""
    ponente = {"name": None, "title": None, "role": None}
    dec_match = PATTERN_DECISION_START.search(text) or PATTERN_RESOLUTION_START.search(text)
    if not dec_match:
        return ponente

    after_dec = text[dec_match.end():dec_match.end() + 500]
    pon_match = PATTERN_PONENTE.search(after_dec)
    if pon_match:
        name = pon_match.group(1).strip()
        title_raw = pon_match.group(2).strip().rstrip('.:')
        title = re.sub(r'\s+', '', title_raw).upper()
        if title in ('CJ', 'C.J', 'C.J.'):
            title = 'C.J.'
        else:
            title = 'J.'
        ponente = {"name": name.title(), "title": title, "role": None}

    return ponente


def _parse_decision_text(text):
    """Extract the main decision body from D E C I S I O N to SO ORDERED."""
    dec_match = PATTERN_DECISION_START.search(text) or PATTERN_RESOLUTION_START.search(text)
    if not dec_match:
        return None

    body = text[dec_match.end():]

    so_match = PATTERN_SO_ORDERED.search(body)
    if so_match:
        body = body[:so_match.end()]

    pon_match = PATTERN_PONENTE.search(body[:500])
    if pon_match:
        body = body[pon_match.end():]

    return _clean_body_text(body)


def _parse_justices(text):
    """Parse justice signatures after SO ORDERED."""
    justices = []
    so_match = PATTERN_SO_ORDERED.search(text)
    if not so_match:
        return justices

    after_so = text[so_match.end():so_match.end() + 2000]

    sep_match = PATTERN_SEPARATE_OPINION.search(after_so)
    if sep_match:
        after_so = after_so[:sep_match.start()]

    justice_pattern = re.compile(
        r'([A-Z][a-z\-]+(?:\s+[A-Z][a-z\-]+)*)\s*,\s*'
        r'(C\.?\s*J\.?|J\.?)\s*'
        r'(?:\(([^)]+)\))?\s*'
        r'(?:,\s*(concur|dissent|on\s+(?:official\s+)?leave|no\s+part|took\s+no\s+part))?',
        re.IGNORECASE
    )

    for m in justice_pattern.finditer(after_so):
        name = m.group(1).strip()
        title = 'C.J.' if 'C' in m.group(2).upper().replace(' ', '') else 'J.'
        role = m.group(3).strip() if m.group(3) else None

        designation = "concurring"
        if m.group(4):
            desg = m.group(4).lower()
            if "dissent" in desg:
                designation = "dissenting"
            elif "leave" in desg:
                designation = "on_leave"
            elif "no part" in desg:
                designation = "no_part"
        else:
            context = after_so[max(0, m.start()-10):m.end()+50].lower()
            if "dissent" in context:
                designation = "dissenting"
            elif "leave" in context:
                designation = "on_leave"
            elif "no part" in context:
                designation = "no_part"

        justices.append({
            "name": name,
            "title": title,
            "role": role,
            "designation": designation,
        })

    return justices


def _parse_separate_opinions(text):
    """Parse separate/concurring/dissenting opinions."""
    opinions = []
    matches = list(PATTERN_SEPARATE_OPINION.finditer(text))

    for i, m in enumerate(matches):
        opinion_type = m.group(1).strip().upper()

        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        opinion_text = text[start:end]

        author = {"name": None, "title": None, "role": None}
        pon_match = PATTERN_PONENTE.search(opinion_text[:500])
        if pon_match:
            author["name"] = pon_match.group(1).strip().title()
            title_raw = re.sub(r'\s+', '', pon_match.group(2)).upper()
            author["title"] = 'C.J.' if 'C' in title_raw else 'J.'

        opinions.append({
            "type": opinion_type,
            "author": author,
            "text": _clean_body_text(opinion_text),
            "concurring_justices": [],
        })

    return opinions


def _parse_footnotes(text):
    """Extract footnote definitions from the case text."""
    footnotes = []
    seen = set()
    for m in PATTERN_FOOTNOTE_DEF.finditer(text):
        num = int(m.group(1))
        if num not in seen and num < 500:
            footnotes.append({"number": num, "text": m.group(2).strip()})
            seen.add(num)
    return footnotes


def _clean_body_text(text):
    """Apply text cleaning rules to decision/opinion text."""
    if not text:
        return text
    text = PATTERN_RUNNING_HEADER.sub('', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = text.split('\n')
    lines = [line.rstrip() for line in lines]
    text = '\n'.join(lines)
    return text.strip()


print("Case parsing functions defined (backtracking-safe regex, timeout helper).")

## Footnote Linking

Detects bare superscript-style integers in body text and replaces them with `[^N]` notation. Builds a footnote map per case.

In [9]:
def link_footnotes(case_dict):
    """
    Annotate footnote references in body text as [^N].
    Builds a footnote_map and updates text fields in-place.
    """
    # Build footnote_map from parsed footnotes
    footnote_map = {fn["number"]: fn["text"] for fn in case_dict.get("footnotes", [])}
    valid_numbers = set(footnote_map.keys())

    if not valid_numbers:
        return case_dict

    def _annotate(text):
        if not text:
            return text
        def _replace_ref(m):
            num = int(m.group(1))
            if num in valid_numbers:
                return f"[^{num}]"
            return m.group(0)
        return PATTERN_FOOTNOTE_REF.sub(_replace_ref, text)

    # Annotate main decision text
    if case_dict.get("main_decision_text"):
        case_dict["main_decision_text"] = _annotate(case_dict["main_decision_text"])

    # Annotate separate opinion texts
    for opinion in case_dict.get("separate_opinions", []):
        if opinion.get("text"):
            opinion["text"] = _annotate(opinion["text"])

    return case_dict


print("Footnote linking defined.")

Footnote linking defined.


## Save to SQLite

Creates the database schema and writes parsed cases, TOC entries, and processing logs with transactional safety.

In [10]:
def init_db(db_path):
    """Create the database and tables if they don't exist."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS cases (
            id                 INTEGER PRIMARY KEY AUTOINCREMENT,
            source_filename    TEXT,
            gr_numbers         TEXT,
            date_of_decision   TEXT,
            division           TEXT,
            ponente_name       TEXT,
            ponente_title      TEXT,
            ponente_role       TEXT,
            parties            TEXT,
            counsel            TEXT,
            justices           TEXT,
            syllabus           TEXT,
            main_decision_text TEXT,
            separate_opinions  TEXT,
            footnotes          TEXT,
            pages              TEXT,
            parse_incomplete   INTEGER DEFAULT 0,
            parse_errors       TEXT,
            date_processed     TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS volume_toc (
            id                 INTEGER PRIMARY KEY AUTOINCREMENT,
            source_filename    TEXT,
            raw_entry          TEXT,
            case_title         TEXT,
            gr_number          TEXT,
            page_number        TEXT,
            date               TEXT,
            matched_to_case    INTEGER DEFAULT 0,
            matched_gr_number  TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS processing_log (
            id                 INTEGER PRIMARY KEY AUTOINCREMENT,
            source_filename    TEXT,
            toc_found          INTEGER DEFAULT 0,
            toc_count          INTEGER DEFAULT 0,
            parsed_count       INTEGER DEFAULT 0,
            matched_count      INTEGER DEFAULT 0,
            toc_only           TEXT,
            parsed_only        TEXT,
            cases_inserted     INTEGER DEFAULT 0,
            cases_updated      INTEGER DEFAULT 0,
            cases_skipped      INTEGER DEFAULT 0,
            parse_errors       TEXT,
            processed_at       TEXT,
            force_refresh      INTEGER DEFAULT 0
        )
    """)

    # Create unique indexes for upsert logic
    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_cases_file_gr
        ON cases (source_filename, gr_numbers)
    """)
    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_toc_file_entry
        ON volume_toc (source_filename, raw_entry)
    """)
    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_proclog_file
        ON processing_log (source_filename)
    """)

    conn.commit()
    return conn


def save_volume(conn, source_filename, cases, toc_entries, validation, force_refresh):
    """Save all data for a single volume in a transaction."""
    cur = conn.cursor()
    inserted = 0
    updated = 0
    skipped = 0
    all_errors = []
    now = datetime.now().isoformat()

    try:
        # --- Delete old data for this volume if force refresh ---
        if force_refresh:
            cur.execute("DELETE FROM cases WHERE source_filename = ?", (source_filename,))
            cur.execute("DELETE FROM volume_toc WHERE source_filename = ?", (source_filename,))
            cur.execute("DELETE FROM processing_log WHERE source_filename = ?", (source_filename,))

        # --- Insert/update cases ---
        for case in cases:
            gr_json = json.dumps(case["gr_numbers"])

            # Check if exists
            cur.execute(
                "SELECT id FROM cases WHERE source_filename = ? AND gr_numbers = ?",
                (source_filename, gr_json)
            )
            existing = cur.fetchone()

            values = (
                source_filename,
                gr_json,
                case.get("date_of_decision"),
                case.get("division"),
                case["ponente"].get("name") if case.get("ponente") else None,
                case["ponente"].get("title") if case.get("ponente") else None,
                case["ponente"].get("role") if case.get("ponente") else None,
                json.dumps(case.get("parties", [])),
                json.dumps(case.get("counsel", [])),
                json.dumps(case.get("justices", [])),
                json.dumps(case.get("syllabus", [])),
                case.get("main_decision_text"),
                json.dumps(case.get("separate_opinions", [])),
                json.dumps(case.get("footnotes", [])),
                json.dumps(case.get("pages", [])),
                1 if case.get("parse_incomplete") else 0,
                json.dumps(case.get("parse_errors", [])),
                now,
            )

            if existing:
                cur.execute("""
                    UPDATE cases SET
                        date_of_decision=?, division=?,
                        ponente_name=?, ponente_title=?, ponente_role=?,
                        parties=?, counsel=?, justices=?, syllabus=?,
                        main_decision_text=?, separate_opinions=?,
                        footnotes=?, pages=?, parse_incomplete=?,
                        parse_errors=?, date_processed=?
                    WHERE source_filename=? AND gr_numbers=?
                """, (
                    values[2], values[3],  # date, division
                    values[4], values[5], values[6],  # ponente
                    values[7], values[8], values[9], values[10],  # parties, counsel, justices, syllabus
                    values[11], values[12],  # decision, opinions
                    values[13], values[14], values[15],  # footnotes, pages, incomplete
                    values[16], values[17],  # errors, date
                    source_filename, gr_json,  # WHERE
                ))
                updated += 1
            else:
                cur.execute("""
                    INSERT INTO cases (
                        source_filename, gr_numbers, date_of_decision, division,
                        ponente_name, ponente_title, ponente_role,
                        parties, counsel, justices, syllabus,
                        main_decision_text, separate_opinions,
                        footnotes, pages, parse_incomplete,
                        parse_errors, date_processed
                    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                """, values)
                inserted += 1

            if case.get("parse_errors"):
                all_errors.append({
                    "gr_number": ", ".join(case.get("gr_numbers", [])),
                    "page_range": f"{case['pages'][0]}-{case['pages'][-1]}" if case.get("pages") else "",
                    "error_message": "; ".join(case["parse_errors"]),
                })

        # --- Insert/update TOC entries ---
        for entry in toc_entries:
            cur.execute("""
                INSERT INTO volume_toc (
                    source_filename, raw_entry, case_title, gr_number,
                    page_number, date, matched_to_case, matched_gr_number
                ) VALUES (?,?,?,?,?,?,?,?)
                ON CONFLICT(source_filename, raw_entry) DO UPDATE SET
                    case_title=excluded.case_title,
                    gr_number=excluded.gr_number,
                    page_number=excluded.page_number,
                    date=excluded.date,
                    matched_to_case=excluded.matched_to_case,
                    matched_gr_number=excluded.matched_gr_number
            """, (
                entry["source_filename"],
                entry["raw_entry"],
                entry.get("case_title"),
                entry.get("gr_number"),
                entry.get("page_number"),
                entry.get("date"),
                entry.get("matched_to_case", 0),
                entry.get("matched_gr_number"),
            ))

        # --- Insert/update processing log ---
        cur.execute("""
            INSERT INTO processing_log (
                source_filename, toc_found, toc_count, parsed_count,
                matched_count, toc_only, parsed_only,
                cases_inserted, cases_updated, cases_skipped,
                parse_errors, processed_at, force_refresh
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
            ON CONFLICT(source_filename) DO UPDATE SET
                toc_found=excluded.toc_found,
                toc_count=excluded.toc_count,
                parsed_count=excluded.parsed_count,
                matched_count=excluded.matched_count,
                toc_only=excluded.toc_only,
                parsed_only=excluded.parsed_only,
                cases_inserted=excluded.cases_inserted,
                cases_updated=excluded.cases_updated,
                cases_skipped=excluded.cases_skipped,
                parse_errors=excluded.parse_errors,
                processed_at=excluded.processed_at,
                force_refresh=excluded.force_refresh
        """, (
            source_filename,
            1 if validation.get("toc_count", 0) > 0 or len(toc_entries) > 0 else 0,
            validation.get("toc_count", 0),
            validation.get("parsed_count", 0),
            validation.get("matched_count", 0),
            json.dumps(validation.get("toc_only", [])),
            json.dumps(validation.get("parsed_only", [])),
            inserted,
            updated,
            skipped,
            json.dumps(all_errors),
            now,
            1 if force_refresh else 0,
        ))

        conn.commit()

    except Exception as e:
        conn.rollback()
        logger.error(f"{source_filename}: DB write failed, rolled back — {e}")
        raise

    return inserted, updated, skipped


print("Database functions defined.")

Database functions defined.


## Main Processing Loop

Processes each queued volume: extract → parse TOC → find boundaries → validate → parse cases → link footnotes → save.

Each step is timed so stalls are immediately visible. Individual case parsing has a 60-second timeout to prevent regex hangs from blocking the entire batch.

In [ ]:
import time

# Per-volume timeout in seconds (skip volumes that take longer than this)
VOLUME_TIMEOUT_SEC = 300  # 5 minutes

conn = init_db(DB_PATH)

total_inserted = 0
total_updated = 0
total_errors = 0
total_skipped_timeout = 0

for pdf_path in tqdm(queued_volumes, desc="Parsing volumes"):
    source_filename = pdf_path.name
    vol_start = time.time()
    logger.info(f"Processing: {source_filename}")

    try:
        # 1. Extract text per page
        t0 = time.time()
        pages = extract_pages(pdf_path, manifest_data=manifest_data)
        t_extract = time.time() - t0
        logger.info(f"  {source_filename}: extract_pages → {len(pages)} pages in {t_extract:.1f}s")

        # 2. Parse TOC
        t0 = time.time()
        toc_found, toc_entries = parse_toc(pages, source_filename)
        t_toc = time.time() - t0

        # 3. Find case boundaries
        t0 = time.time()
        case_boundaries, combined_text = find_case_boundaries(pages)
        t_boundaries = time.time() - t0

        # 4. Validate TOC vs. parsed
        t0 = time.time()
        validation, toc_entries = validate_toc_vs_parsed(
            toc_entries, case_boundaries, source_filename
        )
        if not toc_found:
            validation["toc_count"] = 0
        t_validate = time.time() - t0

        # Print per-volume validation
        print(f"\n{source_filename}: ({len(pages)} pages, extract={t_extract:.1f}s)")
        print(f"  Cases found: {len(case_boundaries)}  "
              f"[TOC={t_toc:.1f}s, boundaries={t_boundaries:.1f}s, validate={t_validate:.1f}s]")
        if toc_found:
            print(f"  TOC entries: {validation['toc_count']}, matched: {validation['matched_count']}")
            if validation["toc_only"]:
                print(f"  TOC only (missing from parse): {validation['toc_only']}")
            if validation["parsed_only"]:
                print(f"  Parsed only (not in TOC):      {validation['parsed_only']}")
        else:
            print("  TOC: not found")

        # 5. Parse each case (with per-case timeout)
        t0 = time.time()
        parsed_cases = []
        for boundary in case_boundaries:
            case_text = combined_text[boundary["start_offset"]:boundary["end_offset"]]
            gr_label = ', '.join(boundary.get('gr_numbers', ['unknown']))
            try:
                case_dict = _run_with_timeout(
                    parse_single_case,
                    args=(case_text, boundary, source_filename),
                    timeout_sec=60,
                )
                case_dict = link_footnotes(case_dict)
                parsed_cases.append(case_dict)
            except _TimeoutError:
                logger.error(
                    f"{source_filename}: Case {gr_label} timed out (>60s), skipping"
                )
                parsed_cases.append({
                    "source_filename": source_filename,
                    "gr_numbers": boundary.get("gr_numbers", []),
                    "date_of_decision": None,
                    "division": boundary.get("division"),
                    "parties": [], "counsel": [],
                    "ponente": {"name": None, "title": None, "role": None},
                    "syllabus": [],
                    "main_decision_text": None,
                    "justices": [], "separate_opinions": [],
                    "footnotes": [],
                    "pages": list(range(boundary.get("start_page", 0), boundary.get("end_page", 0) + 1)),
                    "parse_incomplete": True,
                    "parse_errors": ["Timed out after 60s"],
                })
                total_errors += 1
            except Exception as e:
                logger.error(
                    f"{source_filename}: Failed to parse case {gr_label} "
                    f"(pages {boundary.get('start_page')}-{boundary.get('end_page')}): {e}"
                )
                parsed_cases.append({
                    "source_filename": source_filename,
                    "gr_numbers": boundary.get("gr_numbers", []),
                    "date_of_decision": None,
                    "division": boundary.get("division"),
                    "parties": [], "counsel": [],
                    "ponente": {"name": None, "title": None, "role": None},
                    "syllabus": [],
                    "main_decision_text": None,
                    "justices": [], "separate_opinions": [],
                    "footnotes": [],
                    "pages": list(range(boundary.get("start_page", 0), boundary.get("end_page", 0) + 1)),
                    "parse_incomplete": True,
                    "parse_errors": [str(e)],
                })
                total_errors += 1
        t_parse = time.time() - t0

        # 6. Save to SQLite
        t0 = time.time()
        ins, upd, skp = save_volume(
            conn, source_filename, parsed_cases, toc_entries, validation, FORCE_REFRESH
        )
        t_save = time.time() - t0
        total_inserted += ins
        total_updated += upd

        vol_elapsed = time.time() - vol_start
        print(f"  DB: {ins} inserted, {upd} updated  "
              f"[parse={t_parse:.1f}s, save={t_save:.1f}s, total={vol_elapsed:.1f}s]")

        # Print G.R. numbers for each case
        for c in parsed_cases:
            gr_str = ', '.join(c.get('gr_numbers', []))
            pg = c.get('pages', [])
            pg_str = f"pp.{pg[0]}-{pg[-1]}" if pg else ""
            print(f"    {gr_str}  {pg_str}")

    except Exception as e:
        vol_elapsed = time.time() - vol_start
        logger.error(f"{source_filename}: Volume processing failed after {vol_elapsed:.1f}s — {e}")
        total_errors += 1

print(f"\n{'='*50}")
print(f"Batch complete: {total_inserted} inserted, {total_updated} updated, {total_errors} errors")
print(f"{'='*50}")

## JSON Export

Exports parsed cases per volume as JSON files, including TOC validation data.

In [ ]:
cur = conn.cursor()

# Get all distinct source filenames in the DB
cur.execute("SELECT DISTINCT source_filename FROM cases ORDER BY source_filename")
all_sources = [row[0] for row in cur.fetchall()]

exported = 0
skipped_json = 0

for source_filename in tqdm(all_sources, desc="Exporting JSON"):
    json_path = JSON_EXPORT_PATH / f"{Path(source_filename).stem}.json"

    # Skip if already exists and not force refresh
    if not FORCE_REFRESH and json_path.exists():
        skipped_json += 1
        continue

    # Fetch cases
    cur.execute("SELECT * FROM cases WHERE source_filename = ?", (source_filename,))
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

    cases_list = []
    for row in rows:
        case = dict(zip(columns, row))
        # Parse JSON fields back to objects for clean export
        for field in ["gr_numbers", "parties", "counsel", "justices",
                      "syllabus", "separate_opinions", "footnotes",
                      "pages", "parse_errors"]:
            if case.get(field) and isinstance(case[field], str):
                try:
                    case[field] = json.loads(case[field])
                except json.JSONDecodeError:
                    pass
        cases_list.append(case)

    # Fetch validation info from processing_log
    cur.execute(
        "SELECT toc_found, toc_count, parsed_count, matched_count, toc_only, parsed_only "
        "FROM processing_log WHERE source_filename = ?",
        (source_filename,)
    )
    log_row = cur.fetchone()

    toc_validation = {}
    if log_row:
        toc_validation = {
            "toc_found": bool(log_row[0]),
            "toc_count": log_row[1],
            "parsed_count": log_row[2],
            "matched_count": log_row[3],
            "toc_only": json.loads(log_row[4]) if log_row[4] else [],
            "parsed_only": json.loads(log_row[5]) if log_row[5] else [],
        }

    output = {
        "toc_validation": toc_validation,
        "cases": cases_list,
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False, default=str)

    exported += 1

print(f"JSON export: {exported} files written, {skipped_json} skipped (already exist)")

## Stats Summary & CSV Export

Computes and displays key metrics, then exports to CSV.

In [ ]:
cur = conn.cursor()

def _count(query):
    cur.execute(query)
    return cur.fetchone()[0]

stats = {
    "Total PDFs processed this run": len(queued_volumes),
    "Total cases parsed (all time)": _count("SELECT COUNT(*) FROM cases"),
    "Cases fully parsed (no errors)": _count("SELECT COUNT(*) FROM cases WHERE parse_incomplete = 0"),
    "Cases with parse_incomplete=true": _count("SELECT COUNT(*) FROM cases WHERE parse_incomplete = 1"),
    "Cases missing ponente": _count("SELECT COUNT(*) FROM cases WHERE ponente_name IS NULL OR ponente_name = ''"),
    "Cases missing counsel": _count("SELECT COUNT(*) FROM cases WHERE counsel IS NULL OR counsel = '[]'"),
    "Cases missing date of decision": _count("SELECT COUNT(*) FROM cases WHERE date_of_decision IS NULL OR date_of_decision = ''"),
    "Cases with separate opinions": _count("SELECT COUNT(*) FROM cases WHERE separate_opinions IS NOT NULL AND separate_opinions != '[]'"),
    "Cases with footnotes": _count("SELECT COUNT(*) FROM cases WHERE footnotes IS NOT NULL AND footnotes != '[]'"),
    "Total parse errors logged": _count("SELECT COUNT(*) FROM cases WHERE parse_errors IS NOT NULL AND parse_errors != '[]'"),
    "Volumes with TOC found": _count("SELECT COUNT(*) FROM processing_log WHERE toc_found = 1"),
    "Volumes with TOC discrepancies": _count(
        "SELECT COUNT(*) FROM processing_log WHERE toc_count != matched_count OR "
        "(parsed_only IS NOT NULL AND parsed_only != '[]')"
    ),
    "Total TOC entries across all volumes": _count("SELECT COUNT(*) FROM volume_toc"),
    "Total TOC entries matched to parsed cases": _count("SELECT COUNT(*) FROM volume_toc WHERE matched_to_case = 1"),
}

# Display
print(f"{'='*55}")
print("Parse Statistics")
print(f"{'='*55}")
for metric, value in stats.items():
    print(f"  {metric:<45} {value}")
print(f"{'='*55}")

# Per-volume case counts
print("\nPer-volume case counts:")
cur.execute(
    "SELECT source_filename, COUNT(*) as cnt FROM cases "
    "GROUP BY source_filename ORDER BY source_filename"
)
for row in cur.fetchall():
    print(f"  {row[0]:<35} {row[1]:>5} cases")

# --- CSV export ---
csv_path = STATS_EXPORT_PATH / "parse_stats.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Metric", "Value"])
    for metric, value in stats.items():
        writer.writerow([metric, value])

print(f"\nStats exported to: {csv_path.resolve()}")

## Validation Spot-Check

Displays 3 randomly sampled cases with key fields and their corresponding TOC entries.

In [ ]:
cur.execute("SELECT COUNT(*) FROM cases")
total_cases = cur.fetchone()[0]

if total_cases > 0:
    sample_size = min(3, total_cases)
    cur.execute("SELECT id FROM cases")
    all_ids = [row[0] for row in cur.fetchall()]
    sampled_ids = random.sample(all_ids, sample_size)

    for case_id in sampled_ids:
        cur.execute("SELECT * FROM cases WHERE id = ?", (case_id,))
        columns = [desc[0] for desc in cur.description]
        row = cur.fetchone()
        case = dict(zip(columns, row))

        gr_nums = case.get("gr_numbers", "[]")
        try:
            gr_list = json.loads(gr_nums) if isinstance(gr_nums, str) else gr_nums
        except json.JSONDecodeError:
            gr_list = [gr_nums]

        parties_data = case.get("parties", "[]")
        try:
            parties_list = json.loads(parties_data) if isinstance(parties_data, str) else parties_data
        except json.JSONDecodeError:
            parties_list = []

        footnotes_data = case.get("footnotes", "[]")
        try:
            fn_list = json.loads(footnotes_data) if isinstance(footnotes_data, str) else footnotes_data
        except json.JSONDecodeError:
            fn_list = []

        errors_data = case.get("parse_errors", "[]")
        try:
            err_list = json.loads(errors_data) if isinstance(errors_data, str) else errors_data
        except json.JSONDecodeError:
            err_list = []

        print(f"\n{'='*60}")
        print(f"Case ID:    {case_id}")
        print(f"G.R. Nos:   {', '.join(gr_list)}")
        print(f"Date:       {case.get('date_of_decision', 'N/A')}")
        print(f"Division:   {case.get('division', 'N/A')}")
        print(f"Ponente:    {case.get('ponente_name', 'N/A')} {case.get('ponente_title', '')} {case.get('ponente_role') or ''}")
        print(f"Parties:    {json.dumps(parties_list, indent=2)[:300]}")
        dec_text = case.get("main_decision_text") or ""
        print(f"Decision:   {dec_text[:300]}...")
        print(f"Footnotes:  {len(fn_list)}")
        print(f"Errors:     {err_list if err_list else 'None'}")

        # Show matching TOC entry
        for gr in gr_list:
            cur.execute(
                "SELECT raw_entry, matched_to_case, matched_gr_number "
                "FROM volume_toc "
                "WHERE source_filename = ? AND (matched_gr_number LIKE ? OR gr_number LIKE ?)",
                (case.get("source_filename"), f"%{gr}%", f"%{gr}%")
            )
            toc_rows = cur.fetchall()
            if toc_rows:
                for tr in toc_rows:
                    print(f"  TOC entry:  {tr[0][:120]}")
                    print(f"  Matched:    {'Yes' if tr[1] else 'No'}")
            else:
                print(f"  TOC entry:  (no matching TOC entry found for {gr})")
else:
    print("No cases in database to sample.")

# Close the database connection
conn.close()
print("\nDatabase connection closed.")